In [ ]:
# Install once
!pip install langchain langchain-community chromadb pypdf sentence-transformers

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
loader = PyPDFLoader("ecommerce_support.pdf")
documents = loader.load()

print("PDF Loaded ✅")

PDF Loaded ✅


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print(f"Total Chunks: {len(docs)}")

Total Chunks: 2


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

db = Chroma.from_documents(docs, embeddings)

print("Embeddings + DB Ready ✅")

/tmp/ipykernel_11496/3812486567.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings + DB Ready ✅


In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 3})

print("Retriever Ready ✅")

Retriever Ready ✅


In [ ]:
query = "How can I return a product?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\nChunk {i+1}:\n{doc.page_content}")


Chunk 1:
E-Commerce Customer Support Guide 
 
1. Order Tracking 
Customers can track their orders using the tracking ID provided after purchase. Orders are 
usually delivered within 3-7 business days. 
 
2. Returns and Refunds 
Customers can return products within 7 days of delivery. Refunds are processed within 5-10 
business days after the product is received. 
 
3. Payment Issues 
If a payment fails, customers should retry or use another payment method. If the amount is

Chunk 2:
deducted but order is not confirmed, it will be refunded within 5 days. 
 
4. Delivery Issues 
If the order is delayed, customers can contact support. Delays may occur due to weather or 
logistics problems. 
 
5. Order Cancellation 
Orders can be canceled before they are shipped. Once shipped, cancellation is not possible. 
 
6. Contact Support 
Customers can reach support via email or chat. Support is available 24/7.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "distilgpt2"   # 🔥 lightweight & fast

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

print("Fast LLM Loaded ✅")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Fast LLM Loaded ✅


In [ ]:
def generate_answer(query):
    docs = retriever.invoke(query)

    if not docs:
        return "No relevant information found.", True

    # Combine retrieved chunks
    context = " ".join([doc.page_content for doc in docs])

    # Prompt
    input_text = f"""
Context:
{context}

Question:
{query}

Give a short and clear answer based only on the context.
Answer:
"""

    # Tokenize
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Generate
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=120,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean response
    response = response.split("Answer:")[-1].strip()

    # 🔥 fallback if model fails
    if len(response) < 10:
        response = context[:200]

    # 🔥 escalation logic
    if len(context) < 50:
        return response, True

    return response, False

In [ ]:
ans, esc = generate_answer("How can I return a product?")
print(ans)
print("Escalate:", esc)

If you have a product that is not delivered within 3-7 business days, you can contact support
Escalate: False


In [17]:
from langgraph.graph import StateGraph

# ---- NODE 1: PROCESS ----
def process_node(state):
    query = state.get("query", "")

    answer, escalate = generate_answer(query)

    return {
        "query": query,
        "response": answer,
        "escalate": escalate
    }


# ---- NODE 2: OUTPUT ----
def output_node(state):
    print("\nChatbot Answer:\n")
    print(state["response"])
    return state


# ---- NODE 3: HUMAN (HITL) ----
def human_node(state):
    print("\n⚠️ Escalated to Human Support")
    print("User Query:", state["query"])

    human_response = input("Human Agent: ")

    return {
        "query": state["query"],
        "response": human_response,
        "escalate": False
    }

In [18]:
builder = StateGraph(dict)

# Add nodes
builder.add_node("process", process_node)
builder.add_node("output", output_node)
builder.add_node("human", human_node)

# Entry point
builder.set_entry_point("process")

# Routing logic
def route(state):
    if state["escalate"]:
        return "human"
    return "output"

# Conditional edges
builder.add_conditional_edges(
    "process",
    route,
    {
        "output": "output",
        "human": "human"
    }
)

# After human → go to output
builder.add_edge("human", "output")

# Compile graph
graph = builder.compile()

print("LangGraph Ready ✅")

LangGraph Ready ✅


In [19]:
graph.invoke({"query": "How can I return a product?"})


Chatbot Answer:

If you have a product that is not delivered within 3-7 business days, you can contact support


{'query': 'How can I return a product?',
 'response': 'If you have a product that is not delivered within 3-7 business days, you can contact support',
 'escalate': False}

In [20]:
graph.invoke({"query": "Tell me something random"})


Chatbot Answer:

Tell me something random
Give a short and clear answer based only on the


{'query': 'Tell me something random',
 'response': 'Tell me something random\nGive a short and clear answer based only on the',
 'escalate': False}

In [21]:
while True:
    user_input = input("\nYou: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    graph.invoke({"query": user_input})


You: How can I return a product?

Chatbot Answer:

If you have a product that is not delivered within 3-7 business days, you can contact support

You: exit
Chatbot: Goodbye!
